In [ ]:
# !pip install dm_control mujoco wandb

In [ ]:
import wandb
# # Colab Secrets에서 방금 저장한 API 키 불러오기
# wandb_api_key = userdata.get('WANDB')

# # 불러온 키를 이용해 팝업 없이 백그라운드에서 자동 로그인
# wandb.login(key=wandb_api_key)

In [ ]:
# from google.colab import drive
# drive.mount('/gdrive')

In [ ]:
# Install OSMesa development package for headless rendering
# This is necessary when MUJOCO_GL is set to 'osmesa' and causes AttributeError if missing.
# After running this, please re-execute the cell that sets os.environ['MUJOCO_GL'] and then this cell.
!apt-get update -qq && apt-get install -y libosmesa6-dev

In [ ]:
import os
import numpy as np
os.environ['MUJOCO_GL'] = 'egl'  # GPU 있을 때

from dm_control import suite
from dm_control.suite.wrappers import pixels

In [ ]:
class NormalizeWrapper:
    def __init__(self, env):
        self.env = env
        self.action_spec = env.action_spec

    def reset(self):
        timestep = self.env.reset()
        return self._normalize(timestep)

    def step(self, action):
        timestep = self.env.step(action)
        return self._normalize(timestep)

    def _normalize(self, timestep):
        obs = timestep.observation.copy()
        obs['pixels'] = obs['pixels'].astype(np.float32) / 255.0
        return timestep._replace(observation=obs)

    def last(self):
        return self.env.last()

In [ ]:
import numpy as np
from typing import Union
from torch.utils.data import Dataset, DataLoader
# create random trajectories, and save them as npc files.
from tqdm.notebook import tqdm

def create_trajectory():
    env = suite.load('walker', 'walk', task_kwargs={'random': 42})
    env = pixels.Wrapper(env, render_kwargs={'height': 64, 'width': 64, 'camera_id': 0})
    env = NormalizeWrapper(env)
    action_spec = env.action_spec()
    step = env.reset()
    actions = [np.zeros(6)]
    observations = [step.observation['pixels']]
    rewards = [0.0]
    continuations = [1.0]

    while not step.last():
        action = np.random.uniform(action_spec.minimum, action_spec.maximum)
        step = env.step(action)

        actions.append(action)
        o_t = step.observation['pixels']
        r_t = step.reward
        continuation = step.discount

        observations.append(o_t)
        rewards.append(r_t)
        continuations.append(continuation)

    return {
        'observation': np.array(observations),
        'action': np.array(actions),
        'reward': np.array(rewards),
        'continuation': np.array(continuations)
    }

def save_episodes(save_dir, num_episodes:int = 5):
    os.makedirs(save_dir, exist_ok=True)
    for i in tqdm(range(num_episodes), desc = "generating episodes"):
        episode = create_trajectory()
        np.savez(os.path.join(save_dir, f'{i}.npz'), **episode)

def load_episodes(episode_dir):
    episodes = []
    for file in os.listdir(episode_dir):
        if not file.endswith('npz'):
            continue
        data = np.load(os.path.join(episode_dir, file))
        episode = {k: data[k] for k in data.files}  # 메모리에 실체화
        episodes.append(episode)
    return episodes

def sample_batch(episodes, batch_size: int = 4, sequence_length: int = 50):
    batch = []
    for _ in range(batch_size):
        episode = episodes[np.random.choice(len(episodes))]
        len_episode = episode['observation'].shape[0]
        start_idx = np.random.randint(0, len_episode - sequence_length)
        end_idx = start_idx + sequence_length
        new_episode = {
            'observation': episode['observation'][start_idx:end_idx],
            'action': episode['action'][start_idx:end_idx],
            'reward': episode['reward'][start_idx:end_idx],
            'continuation': episode['continuation'][start_idx:end_idx]
        }
        batch.append(new_episode)

    batch_tensor = {
        'observation': torch.from_numpy(np.stack([episode['observation'] for episode in batch])),
        'action': torch.from_numpy(np.stack([episode['action'] for episode in batch])).float(),
        'reward': torch.from_numpy(np.stack([episode['reward'] for episode in batch])).float(),
        'continuation': torch.from_numpy(np.stack([episode['continuation'] for episode in batch])).float()
    }
    return batch_tensor


In [ ]:
from torch import nn
import torch
import torch.nn.functional as F

def kl_divergence(mu_1, sigma_1, mu_2, sigma_2):
    # shape: (B, T, D)
    return (0.5*(2*(sigma_2.log() - sigma_1.log())
    + (sigma_1.pow(2)
    + (mu_1 - mu_2).pow(2))/sigma_2.pow(2) - 1
     ).sum(dim=-1).mean())

In [ ]:
class Encoder(nn.Module):
    def __init__(self, ):
        """
        obs: (B, 3, 64, 64)
        return: (B, 1024,)
        """
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 32, kernel_size = 4, stride = 2, )
        self.conv2 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size =4, stride = 2)
        self.conv4 = nn.Conv2d(in_channels = 128, out_channels = 256, kernel_size=4, stride = 2)
        self.act = nn.ReLU()

    def forward(self, obs):
        """
        obs: (B, C,H,W) C=3, H=64, W=64
        return: (B, D) D = 1024
        """
        x = obs
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = self.act(self.conv3(x))
        x = self.act(self.conv4(x))
        x = x.reshape(x.size(0), -1) # Flatten
        return x

class Decoder(nn.Module):
    def __init__(self, state_dim: int = 230):
        super().__init__()
        self.linear1 = nn.Linear(state_dim, 1024)
        self.deconv1 = nn.ConvTranspose2d(in_channels = 1024, out_channels = 128,
                                          kernel_size=5,stride=2)
        self.deconv2 = nn.ConvTranspose2d(in_channels = 128, out_channels = 64, kernel_size=5, stride=2)
        self.deconv3 = nn.ConvTranspose2d(in_channels = 64, out_channels = 32, kernel_size=6, stride=2)
        self.deconv4 = nn.ConvTranspose2d(in_channels = 32, out_channels = 3, kernel_size=6, stride=2)
        self.act = nn.ReLU()

    def forward(self, x):
        """
        x: (B, D)
        """
        x = self.act(self.linear1(x)).view(x.size(0), 1024,1,1)
        x = self.act(self.deconv1(x))
        x = self.act(self.deconv2(x))
        x = self.act(self.deconv3(x))
        o = self.deconv4(x)
        return o

class ActionModel(nn.Module):
    def __init__(self, num_layers: int = 3, hidden_dim: int = 300, latent_dim: int = 230, action_dim:int = 6):
        super().__init__()
        linear_1 = nn.Linear(latent_dim, hidden_dim)
        linear_last = nn.Linear(hidden_dim, 2*action_dim)
        self.act = nn.ELU()
        layers = []
        layers.append(linear_1)
        layers.append(self.act)
        for _ in range(num_layers-2):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(self.act)
        layers.append(linear_last)
        self.layers = nn.Sequential(*layers)
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.num_layers = num_layers
        self.action_dim = action_dim


    def forward(self, x):
        """
        x: (B, latent_dim), latent_dim is dim_h + dim_z
        return: (B, 2*action_dim ), which is mean_action and variance of action
        """
        raw_statistics = self.layers(x)
        mean, raw_std = torch.chunk(raw_statistics, 2, dim=-1)
        mean = 5*torch.tanh(mean/5)
        std = F.softplus(raw_std) + 1e-4
        eps = torch.randn_like(mean)
        action = torch.tanh(mean + std * eps)
        return action

class ValueModel(nn.Module):
    def __init__(self, num_layers: int = 3, hidden_dim: int = 300, latent_dim: int = 230):
        super().__init__()
        linear_1 = nn.Linear(latent_dim, hidden_dim)
        linear_last = nn.Linear(hidden_dim, 1)
        self.act = nn.ELU()
        layers = []
        layers.append(linear_1)
        layers.append(self.act)
        for _ in range(num_layers-2):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(self.act)
        layers.append(linear_last)
        self.layers = nn.Sequential(*layers)
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.num_layers = num_layers

    def forward(self, x):
        """
        x: (B, latent_dim)
        return: (B, 1)
        """
        return self.layers(x)

class RSSM(nn.Module):
    def __init__(self, hidden_dim:int = 200, z_dim: int = 30, action_dim: int = 6):
        super().__init__()
        self.gru = nn.GRUCell(input_size=z_dim + action_dim, hidden_size=hidden_dim)
        self.posterior_mlp = nn.Sequential(
            nn.Linear(hidden_dim+1024, 300),
            nn.ELU(),
            nn.Linear(300, 300),
            nn.ELU(),
            nn.Linear(300, 2*z_dim)
        )

        self.prior_mlp = nn.Sequential(
            nn.Linear(hidden_dim, 300),
            nn.ELU(),
            nn.Linear(300, 300),
            nn.ELU(),
            nn.Linear(300, 2*z_dim)
        )
        self.softplus = nn.Softplus()
        self.hidden_dim = hidden_dim
        self.z_dim = z_dim



    def forward(self, h, z, action, obs=None, return_dist:bool = False,
                return_both:bool = False):
        """
        if return_both, return both posterior states and prior states
        if return dist: return mean and stds with the states.

        h: (B, D)
        z: (B, D)
        action: (B, A)
        obs: (B, C, H, W) raw image
        """


        h = self.gru(torch.cat([z, action], dim = 1), h)
        if obs is None:
            mean, raw_std = torch.chunk(self.prior_mlp(h), 2, dim=-1)
        else:
            mean, raw_std = torch.chunk(
                self.posterior_mlp(torch.cat([h, obs], dim = 1)), 2, dim=-1)

        std = self.softplus(raw_std) + 1e-4
        eps = torch.randn_like(mean)
        z = mean + std * eps

        if return_both:
            prior_mean, prior_raw_std = torch.chunk(self.prior_mlp(h), 2, dim=-1)
            prior_std = self.softplus(prior_raw_std) + 1e-4
            prior_eps = torch.randn_like(prior_mean)
            prior_z = prior_mean + prior_std * prior_eps
            return h, z, mean, std, prior_z, prior_mean, prior_std

        if return_dist:
            return h, z, mean, std
        else:
            return h, z

class RewardModel(nn.Module):
    def __init__(self, state_dim:int = 230):
        super().__init__()

        self.ffn = nn.Sequential(
            nn.Linear(state_dim, 300),
            nn.ELU(),
            nn.Linear(300, 300),
            nn.ELU(),
            nn.Linear(300, 1)
        )
    def forward(self, s):
        """
        s: (B, D)
        return: (B, 1)
        """
        return self.ffn(s)

In [ ]:
class Dreamer(nn.Module):
    def __init__(self, hidden_dim:int = 200, z_dim:int = 30, action_dim:int = 6, num_ffn_layers: int = 3, discount_factor: float = 0.9999):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder(state_dim = hidden_dim + z_dim)
        self.rssm = RSSM(hidden_dim, z_dim, action_dim)
        self.reward_model = RewardModel(hidden_dim + z_dim)
        self.policy_model = ActionModel(num_layers = num_ffn_layers, hidden_dim = hidden_dim, latent_dim = hidden_dim + z_dim, action_dim = action_dim)
        self.value_model = ValueModel(num_layers = num_ffn_layers, hidden_dim = hidden_dim, latent_dim = hidden_dim + z_dim)

        self.hidden_dim = hidden_dim
        self.z_dim = z_dim
        self.action_dim = action_dim
        self.num_ffn_layers = num_ffn_layers
        self.discount_factor = discount_factor

    def get_dynamics_model_parameters(self):
        return list(self.encoder.parameters()) + list(self.decoder.parameters()) + list(self.rssm.parameters()) + list(self.reward_model.parameters())

    def get_action_model_parameters(self):
        return list(self.policy_model.parameters())

    def get_value_model_parameters(self):
        return list(self.value_model.parameters())

    def get_reward_model_parameters(self):
        return list(self.reward_model.parameters())

    def compute_dynamics_loss(self, actions: torch.Tensor, observations:torch.Tensor,
                      rewards:torch.Tensor, mode = 'reconstruction', beta:float = 1.0):
        """
        Compute dynamics model loss and return computed states.

        Args:
        actions: (B, T, A)
        observations: (B, T, C, H, W)
        rewards: (B, T, 1)
        mode: the mode of dynamics modes. there are 3 modes in the paper.
        beta(float)

        Return
        loss: (scalar)
        states: list of states. [(post_h, post_z, prior_z)]

        """
        num_steps = actions.shape[1]
        batch_size = actions.shape[0]
        height = observations.shape[-2]
        width = observations.shape[-1]
        channels = observations.shape[2]
        device = actions.device

        # 1. compute states
        post_h = torch.zeros(batch_size, self.hidden_dim).to(device)
        post_z = torch.zeros(batch_size, self.z_dim).to(device)
        prior_z = torch.zeros(batch_size, self.z_dim).to(device)
        first_action = torch.zeros(batch_size, self.action_dim).to(device)

        states = []
        prior_dists = []
        posterior_dists = []

        # Encode observations
        obs_encodes = self.encoder(observations.reshape(-1, observations.shape[-3], observations.shape[-2], observations.shape[-1]))
        # observation (B, T, C, H, W) -> (B*T, C, H, W) -> (B*T, D)
        obs_encodes = obs_encodes.reshape(observations.shape[0], observations.shape[1], -1)
        # burn-in rollout

        (post_h, post_z, post_mean,
            post_sigma, prior_z, prior_mean, prior_sigma) = self.rssm(
                h=post_h, z=post_z, action = first_action,
                                    obs = obs_encodes[:, 0, :], return_dist=True,
                                    return_both = True)
        states.append((post_h, post_z, prior_z))
        prior_dists.append((prior_mean, prior_sigma))
        posterior_dists.append((post_mean, post_sigma))


        # Roll Out
        for t in range(1,num_steps,1):
            (post_h, post_z, post_mean,
             post_sigma, prior_z, prior_mean, prior_sigma) = self.rssm(
                 h=post_h, z=post_z, action = actions[:,t-1,:],
                                       obs = obs_encodes[:, t, :], return_dist=True,
                                       return_both = True)
            states.append((post_h, post_z, prior_z))
            prior_dists.append((prior_mean, prior_sigma))
            posterior_dists.append((post_mean, post_sigma))

        # 2. compute loss
        if mode == 'reconstruction':
            # All three start from index 1, at total num_steps - 1 samples

            # Decode states
            hts = torch.stack([s[0] for s in states], dim = 1) # (B, T, D_h)
            zts = torch.stack([s[1] for s in states], dim = 1) # (B, T, D_z)
            states = torch.cat([hts, zts], dim = -1) # (B, T, D_h + D_z)
            states = states.reshape(-1, states.shape[-1])
            obs_decodeds = self.decoder(states).reshape(batch_size, num_steps, channels, height, width)
            reward_preds = self.reward_model(states).reshape(batch_size, num_steps, -1)
            states = states.reshape(batch_size, num_steps, -1)

            # compute losses

            post_means = [m for m,_ in posterior_dists] # length num_steps-1 list of (b,30) tensor
            post_stds = [s for _,s in posterior_dists]
            prior_means = [m for m,_ in prior_dists]
            prior_stds = [s for _,s in prior_dists]

            post_means = torch.stack(post_means, dim = 0)
            post_stds = torch.stack(post_stds, dim = 0)
            prior_means = torch.stack(prior_means, dim = 0)
            prior_stds = torch.stack(prior_stds, dim = 0)

            # observation loss is MSE loss
            observation_loss = F.mse_loss(observations, obs_decodeds)
            reward_loss = F.mse_loss(rewards.unsqueeze(-1), reward_preds)
            kld = kl_divergence(post_means, post_stds, prior_means, prior_stds)
            dynamics_loss = torch.maximum(kld, torch.tensor(3.0, device=kld.device))
            loss = observation_loss + reward_loss + beta*dynamics_loss

            metrics = {
                'Loss/Observation': observation_loss.item(),
                'Loss/Reward': reward_loss.item(),
                'Loss/KLD': kld.item(),
                'Loss/Total': loss.item()
            }

            # package states to return
            # print(f"obs_loss: {observation_loss:.4f} | rew_loss: {reward_loss:.4f} | kld: {kld:.4f}")
            return loss, hts, zts, metrics
        else:
            raise NotImplementedError("Todo")


In [ ]:
def compute_MC_value(rewards, discount_factor: float = 0.99  ):
    """
    estimates Monte Carlo Value estimates

    rewards: (B, H, 1)


    return: (B, H, 1)
    """
    batch_size = rewards.shape[0]
    num_steps = rewards.shape[1]
    estims = torch.zeros(batch_size, num_steps,1).to(rewards.device)
    for t in range(num_steps):
        # discounted sum of rewards
        for i in range(t, num_steps):
            estims[:,t,] = estims[:,t,:] + (discount_factor ** (i-t)) * rewards[:,i]

    return estims

def compute_k_step_TD_value(rewards, values, k = 1,  discount_factor: float = 0.99):
    """
    estimates k-step Temporal Difference estimates of values.


    rewards: (B, H, 1)
    values: (B, H, 1)

    return: (B, H, 1)
    """
    batch_size = rewards.shape[0]
    num_steps = rewards.shape[1]
    estims = torch.zeros(batch_size, num_steps, 1).to(values.device)

    for t in range(num_steps):
        h = min(t+k, num_steps-1)
        for i in range(t, h):
            estims[:,t,] = estims[:,t,] + (discount_factor ** (i-t)) * rewards[:,i]

        estims[:,t,] = estims[:,t,]+ (discount_factor ** (h-t)) * values[:,h,]
    return estims

# @torch.jit.script
def compute_lambda_value(rewards, values, l:float = 0.95, discount_factor: float = 0.99):
    """
    estimates lambda  return estimates of values.


    rewards: (B, H, 1)
    values: (B, H, 1)
    l: lambda value
    return: (B, H, 1)
    """
    batch_size = rewards.shape[0]
    num_steps = rewards.shape[1]

    estims = torch.zeros(batch_size, num_steps, 1).to(values.device)
    estims[:, -1, ] = values[:, -1,]
    for t in range(num_steps-2, -1, -1):
        estims[:, t,] = rewards[:,t] + discount_factor*((1-l)*values[:, t+1]
                                                                + l*estims[:, t+1,])

    return estims

In [ ]:
def init_env(env_name, task_name):
    env = suite.load(env_name, task_name, task_kwargs={'random': 42})
    env = pixels.Wrapper(env, render_kwargs={'height': 64, 'width': 64, 'camera_id': 0})
    env = NormalizeWrapper(env)
    return env

In [ ]:
import glob

def load_checkpoint(ckpt_path):
    # if ckpt path is a directory, then load latest checkpoint
    if os.path.isdir(ckpt_path):
        ckpts = glob.glob(os.path.join(ckpt_path, '*.pt'))
        # the name will have x_dreamer.pt, and x will be the number of iterations. so pick the pt file with max x
        ckpt_path = max(ckpts, key=lambda x: int(x.split('/')[-1].split('_')[0]))
        print(f"Loading checkpoint from {ckpt_path}")
    checkpoint = torch.load(ckpt_path)
    return checkpoint

In [ ]:
import shutil
from tqdm.notebook import tqdm
from pathlib import Path

def train_dreamer(args):

    exp_dir = Path(args.exp_dir)
    ckpt_dir = exp_dir / 'checkpoints'
    dataset_dir = exp_dir / 'dataset'

    if exp_dir.exists():
        y = input(f'{exp_dir} already exists. Go on? (y/n): ')
        if y.lower() == 'n':
            return
    else:
        # parents=True면 중간 경로가 없어도 다 만들어줍니다.
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        dataset_dir.mkdir(parents=True, exist_ok=True)

    print(f"Dataset Directory: {dataset_dir}")

    # 2. 데이터셋 로드 (문자열 경로 대신 Path 객체 전달)
    if not any(dataset_dir.iterdir()): # 폴더가 비어있다면
        save_episodes(dataset_dir, args.num_seed_episodes)
        print("Seed dataset generated.")

    episodes = load_episodes(dataset_dir)
    print("Seed dataset loaded.")

    # initialize models

    env = init_env('walker', 'walk')
    action_bound_min = torch.from_numpy(env.action_spec().minimum.copy()).to(args.device)
    action_bound_max = torch.from_numpy(env.action_spec().maximum.copy()).to(args.device)
    print("environment initialized.")

    device = args.device

    dreamer = Dreamer(hidden_dim = args.hidden_dim, z_dim = args.z_dim, action_dim = args.action_dim, num_ffn_layers = args.num_ffn_layers,
                      discount_factor = args.discount_factor).to(args.device)


    dynamics_optimizer = torch.optim.Adam(dreamer.get_dynamics_model_parameters(),
                                          lr = args.dynamics_model_lr)
    action_optimizer = torch.optim.Adam(dreamer.get_action_model_parameters(),
                                         lr = args.action_model_lr)
    value_optimizer = torch.optim.Adam(dreamer.get_value_model_parameters(),
                                       lr = args.value_model_lr)

    # Load and Initialize
    wandb_run_id = None
    model_path = None
    if args.resume:
        ckpts = list(ckpt_dir.glob('*.pt'))
        if ckpts:
            ckpt_path = max(ckpts, key=lambda x: int(x.stem.split('_')[0]))
            model_path = ckpt_path
            print(f"Loading checkpoint from {ckpt_path}")

            ckpt = torch.load(model_path, weights_only=False)
            dreamer.load_state_dict(ckpt['model_state_dict'])
            dynamics_optimizer.load_state_dict(ckpt['dynamics_optimizer_state_dict'])
            action_optimizer.load_state_dict(ckpt['action_optimizer_state_dict'])
            value_optimizer.load_state_dict(ckpt['value_optimizer_state_dict'])
            start_step = ckpt['num_interactions']
            episode_rewards = ckpt['episode_rewards']
            dynamics_losses = ckpt['dynamics_losses']
            actor_losses = ckpt['actor_losses']
            value_losses = ckpt['value_losses']
            wandb_run_id = ckpt.get('wandb_run_id', None)
            print("Model loaded.")
        else:
            print("No checkpoint found.")
            start_step = 0
            episode_rewards = []
            dynamics_losses = []
            actor_losses = []
            value_losses = []
    else:
        print("Do not resume")
        start_step = 0
        episode_rewards = []
        dynamics_losses = []
        actor_losses = []
        value_losses = []


    wandb.init(
            project="dreamerV1",
            config=vars(args),
            id=wandb_run_id,         # 기존 ID가 있다면 전달, None이면 무시됨
            resume="allow"     # "allow": ID가 일치하면 이어서 쓰고, 없으면 새로 만듦
        )
    wandb.watch(dreamer, log='all', log_freq=100)
    print("Start training loop...")

    outer_pbar = tqdm(
        range(start_step, args.max_steps, 1),
        initial = start_step,
        total = args.max_steps,
        desc="[Environment Interaction Loop]")
    for num_interactions in outer_pbar:

        inner_pbar = tqdm(range(args.collect_interval), desc = "[Dynamics model Training Loop]", leave = False)
        for c in inner_pbar:
            dreamer.train()
            episodes_batch = sample_batch(episodes, args.batch_size, args.seq_length)
            actions = episodes_batch['action'].to(device)
            observations = episodes_batch['observation'].to(device).permute(0,1,4,2,3) # (B,T,H,W,C ) -> (B,T,C,H,W)
            rewards = episodes_batch['reward'].to(device)
            continuations = episodes_batch['continuation'].to(device)

            # compute loss
            dynamics_loss, hts, zts, metrics = dreamer.compute_dynamics_loss(actions, observations, rewards, beta=args.beta)
            dynamics_optimizer.zero_grad()
            dynamics_loss.backward()
            dynamics_optimizer.step()

            dynamics_losses.append(dynamics_loss.item())

            for p in dreamer.get_dynamics_model_parameters():
                p.requires_grad_(False)

            ### TODO: Loop Debugging and modification 필요
            hts = hts.detach()
            zts = zts.detach()
            states_ = torch.cat([hts, zts], dim = -1).reshape(args.batch_size*args.seq_length, -1)
            hts_ = hts.reshape(args.batch_size*args.seq_length, -1)
            zts_ = zts.reshape(args.batch_size*args.seq_length, -1)

            rts = dreamer.reward_model(states_)
            vts = dreamer.value_model(states_)

            rollout_values = [vts]
            rollout_rewards = [rts]

            for _ in range(args.imagination_horizon):
                ats = dreamer.policy_model(states_)
                hts_, zts_ = dreamer.rssm(h=hts_, z=zts_, action=ats)
                rts = dreamer.reward_model(torch.cat([hts_, zts_], dim = 1))
                vts = dreamer.value_model(torch.cat([hts_, zts_], dim = 1))
                rollout_values.append(vts)
                rollout_rewards.append(rts)
                states_ = torch.cat([hts_, zts_], dim = -1)

            rewards = torch.stack(rollout_rewards, dim = 1) # (BT,H, 1)
            values = torch.stack(rollout_values, dim = 1) # (BT,H, 1)

            # lambda estimate
            lambda_values = compute_lambda_value(rewards, values, l=args.lambda_)

            # update action and value models
            action_loss = -torch.mean(lambda_values)
            value_loss = (1/2)*torch.mean((lambda_values.detach() - values)**2)

            action_loss.backward(retain_graph=True)
            value_loss.backward()


            torch.nn.utils.clip_grad_norm_(dreamer.get_action_model_parameters(), 100)
            torch.nn.utils.clip_grad_norm_(dreamer.get_value_model_parameters(), 100)
            action_optimizer.step()
            action_optimizer.zero_grad()

            value_optimizer.step()
            value_optimizer.zero_grad()

            actor_losses.append(action_loss.item())
            value_losses.append(value_loss.item())

            for p in dreamer.get_dynamics_model_parameters():
                p.requires_grad_(True)

            global_step = num_interactions * args.collect_interval + c + 1
            wandb.log(
                {
                    'Loss/Observation': metrics['Loss/Observation'],
                    'Loss/Reward': metrics['Loss/Reward'],
                    'Loss/KLD': metrics['Loss/KLD'],
                    'Loss/Total': metrics['Loss/Total'],
                    'Loss/Action': action_loss.item(),
                    'Loss/Value': value_loss.item(),
                }, step = global_step
            )
            inner_pbar.set_postfix({
                'Loss/Observation': metrics['Loss/Observation'],
                'Loss/Reward': metrics['Loss/Reward'],
                'Loss/KLD': metrics['Loss/KLD'],
                'Loss/Total': metrics['Loss/Total'],
                'Loss/Action': action_loss.item(),
                'Loss/Value': value_loss.item(),
            })

        # Environment Interaction
        step = env.reset()
        ht = torch.zeros(1, args.hidden_dim).to(device)
        zt = torch.zeros(1, args.z_dim).to(device)
        ot = step.observation['pixels'] # (H,W,C)
        observations = [ot]
        actions = [np.zeros(args.action_dim)]
        rewards = [0.0]
        continuations = [1.0]

        ot = torch.from_numpy(ot).permute(2,0,1).unsqueeze(0).to(device) # (1,C,H,W)
        rt = 0.0
        ct = 1.0
        at = torch.zeros(1,6).to(device)

        print("Start Environment Interaction")
        with torch.no_grad():
            while not step.last():
                obs_enc = dreamer.encoder(ot)
                ht, zt = dreamer.rssm(h=ht, z=zt, action=at, obs=obs_enc)
                at = dreamer.policy_model(torch.cat([ht, zt], dim=1))
                exploration_noise = torch.randn_like(at) * args.exploration_noise_scale
                at = (at + exploration_noise).clamp(action_bound_min, action_bound_max).float()

                # action repeat
                repeat_reward = 0.0
                for _ in range(args.action_repeat):
                    step = env.step(at.cpu().squeeze(0).numpy())
                    repeat_reward += step.reward
                    if step.last():
                        break

                ot = step.observation['pixels']
                observations.append(ot)
                actions.append(at.cpu().squeeze(0).numpy())
                rewards.append(repeat_reward)  # 합산된 reward
                continuations.append(step.discount)
                ot = torch.from_numpy(ot).permute(2,0,1).unsqueeze(0).to(device)

        episode_reward = np.array(rewards).sum()
        episode_rewards.append(episode_reward)
        current_total_steps = (num_interactions + 1) * args.collect_interval
        wandb.log(
            {
                'Episode Return': episode_reward
            }, step = current_total_steps
        )
        # save as npz
        new_episode = {
            'observation': np.array(observations),
            'action': np.array(actions),
            'reward': np.array(rewards),
            'continuation': np.array(continuations)
        }
        episodes.append(new_episode)
        np.savez(os.path.join(dataset_dir, f'episode_{len(episodes)}.npz'), **new_episode)

        # Save model parameter
        if num_interactions % args.checkpoint_save_interval == 0:
            torch.save({
                'args': vars(args),  # dict으로 변환해서 저장
                'model_state_dict': dreamer.state_dict(),
                'dynamics_optimizer_state_dict': dynamics_optimizer.state_dict(),
                'action_optimizer_state_dict': action_optimizer.state_dict(),
                'value_optimizer_state_dict': value_optimizer.state_dict(),
                'num_interactions': num_interactions,
                'episode_rewards': episode_rewards,
                'dynamics_losses': dynamics_losses,
                'actor_losses': actor_losses,
                'value_losses': value_losses,
                'wandb_run_id': wandb.run.id if wandb.run else None,
            }, ckpt_dir / f'{num_interactions}_dreamer.pt')


        # torch.save(dreamer.state_dict(), os.path.join(args.project_dir, f'{str(num_interactions)}_dreamer.pt'))
        outer_pbar.set_postfix(
            {
                'Episode Return': episode_reward,
            }
        )
        # print(f"{str(num_interactions)} environment interaction. Episode reward: {episode_reward}")


In [ ]:
from types import SimpleNamespace

default_args = SimpleNamespace(
    # 경로
    exp_dir = '/gdrive/MyDrive/대학원석사입학/Research/SWM/dreamer-v1/0409/',
    resume = True,
    # 환경
    env_name='walker',
    task_name='walk',
    # 모델
    hidden_dim=200,
    z_dim=30,
    action_dim=6,
    num_ffn_layers=3,
    discount_factor=0.99,
    # 학습
    batch_size=50,
    seq_length=50,
    imagination_horizon=15,
    dynamics_model_lr=6e-4,
    action_model_lr=8e-5,
    value_model_lr=8e-5,
    beta=1.0,
    gamma=0.99,
    lambda_=0.95,
    free_nats=3.0,
    # 루프
    max_steps=1000,
    collect_interval=100,
    num_seed_episodes=5,
    exploration_noise_scale=0.3,
    action_repeat=2,
    device=torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'),
    checkpoint_save_interval=1,
)

In [ ]:
args = default_args  # 기본값 그대로 쓰거나
args.collect_interval = 100

train_dreamer(args)

## TODOs


*   Debug codes
*   Add training metrics

